# Enterprise Track 4 · A Scheduled Job That Pages Someone

**Goal:** run the pipeline the way production would: as a scheduled **job**, followed by a quality check that fails loudly, with failures going to email and to PagerDuty or ServiceNow. Then break it on purpose, watch the alert, and resolve it the way the data owner would.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 4

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Jobs and alerts")

## 1. The job
A **Lakeflow job** is a set of tasks with dependencies, a schedule, notifications and a run-as identity. This one has two tasks:

```
refresh_pipeline  ──▶  validate_quality
(pipeline task)        (notebook: enterprise/tasks/validate_quality)
```

`validate_quality` checks the tables after every refresh, appends one row per check to `quality_results`, and **raises an error if any check fails**. A failed task fails the job, and a failed job notifies everyone on its failure list: your email, plus any destinations named in `ALERT_DESTINATIONS`. That's the whole alerting design: checks that fail loudly, and a platform that turns a failure into a page.

The schedule is created **paused**, so nothing runs until you decide.

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
pipeline_id = ent.ensure_pipeline(N)
job_id = ent.ensure_job(N, pipeline_id)
print(f"Open it: {w.config.host}/jobs/{job_id}")

In [ ]:
print((COURSE / "enterprise" / "tasks" / "validate_quality.ipynb").exists() and "The checks and their thresholds are at the top of enterprise/tasks/validate_quality.")
run = ent.run_job(job_id)

In [ ]:
display(ent.sql(f"SELECT * FROM {N['fq']}.quality_results WHERE run_id = '{run.run_id}' ORDER BY status, check_name"))

## 2. Break it: a delivery with broken IDs
The source system's export breaks and three TDCJ numbers arrive with a digit missing. The pipeline doesn't stop: the drop expectation keeps those rows out of silver, the quarantine table holds them as received, and every other row flows through. Then `validate_quality` finds quarantined rows nobody has reviewed and fails the job, which sends the alert.

In [ ]:
ent.land_batch(N, 3, bad_ids=3)
run = ent.run_job(job_id)            # expect FAILED: that's the point

In [ ]:
display(ent.sql(f"SELECT _id_problem, tdcj_number, sid_number, last_name, _source_file FROM {N['fq']}.silver_quarantine"))
display(ent.sql(f"SELECT check_name, value, threshold, status, detail FROM {N['fq']}.quality_results WHERE run_id = '{run.run_id}' AND status = 'FAIL'"))

Check your email: the failure notification links to the run. If an admin set up a PagerDuty destination and you named it in `ALERT_DESTINATIONS`, an incident opened too.

## 3. Resolve it the way the data owner would
Nothing is fixed by guessing. The data owner decides what happens to each held record, and the decision is recorded: here, the three records go back to the source for correction. Recording the review is what clears the alert; the corrected records arrive in a later delivery like any other.

In [ ]:
ent.sql(f"""
    INSERT INTO {N['fq']}.quarantine_reviews
    SELECT _source_file, tdcj_number, 'returned to source for correction', current_user(), current_timestamp()
    FROM {N['fq']}.silver_quarantine q
    WHERE NOT EXISTS (SELECT 1 FROM {N['fq']}.quarantine_reviews r
                      WHERE r.source_file = q._source_file AND r.tdcj_number_as_written = q.tdcj_number)""")
ent.move_to_rejected(N, "batch_003.csv")     # keep the bad delivery for the data owner, out of the landing zone
run = ent.run_job(job_id)                    # expect SUCCESS

## 4. Turn the schedule on (when you're ready)
The job runs daily at 5:50 AM Central once unpaused. Leave it paused while you're learning; a scheduled job that fails every morning is noise, and noise trains people to ignore alerts.

In [ ]:
from databricks.sdk.service import jobs as J
UNPAUSE = False      # change to True, run, and the schedule starts
if UNPAUSE:
    settings = ent.job_settings(N, pipeline_id)
    settings["schedule"].pause_status = J.PauseStatus.UNPAUSED
    w.jobs.reset(job_id, new_settings=J.JobSettings(**settings))
print("Schedule:", w.jobs.get(job_id).settings.schedule.pause_status.value)

## Getting alerts to the right place
| Where | How | Notes |
|---|---|---|
| Email | the job's failure list | works everywhere; set `ALERT_EMAILS` |
| PagerDuty | a notification destination with an Events API v2 integration key | admin creates the destination once; any job can use it |
| ServiceNow, Vivantio | a webhook destination | the receiving side needs an inbound endpoint (a ServiceNow Scripted REST API or IntegrationHub flow) that turns the webhook JSON into an incident |
| Prometheus and Grafana | read `quality_results` and the system tables | see notebook 05 |

**Next:** 05_monitoring adds monitoring that doesn't depend on anyone writing a check.